# 오늘 연습

2760개 기업의 3개월 전 주가가 필요

전날 주가와 비교하여 가장 많이 상승한 기업 10개를 추출

1. 필요 데이터
    - 코드, 기업이름, 주가변화량
    - 주가 변화량 = 전일 주가 - 3개월전 주가


In [282]:
# streamlit 설치
!pip install streamlit

In [283]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import streamlit as st

In [284]:
code_all = pd.read_html('http://kind.krx.co.kr/corpgeneral/corpList.do?method=download', header=0, encoding='euc-kr')

/usr/local/lib/python3.12/dist-packages/bs4/__init__.py:341: UserWarning: You provided Unicode markup but also provided a value for from_encoding. Your from_encoding will be ignored.
  warnings.warn(


In [285]:
df = code_all[0]

In [286]:
df.head()

,회사명,시장구분,종목코드,업종,주요제품,상장일,결산월,대표자명,홈페이지,지역
0,보원케미칼,코스닥,0010F0,플라스틱제품 제조업,"차량 내장재용 표면소재, SPC바닥재 등",2026-04-03,12월,허찬회,http://www.bowonchem.co.kr,충청북도
1,교보20호스팩,코스닥,0132G0,기타 금융업,기업 인수 및 합병,2026-04-02,12월,김서호,NaN,서울특별시
2,인벤테라,코스닥,0007J0,자연과학 및 공학 연구개발업,철(Fe) 기반 질환 특이성 나노-MRI 조영제 신약,2026-04-02,12월,신태현,http://inventera.kr,서울특별시
3,신한제17호스팩,코스닥,0130D0,금융 지원 서비스업,금융지원 서비스업,2026-04-01,12월,이효상,NaN,서울특별시
4,리센스메디컬,코스닥,394420,의료용 기기 제조업,"피부 냉각마취 의료기기, 경피약물전달 의료기기, 동물용 경피약물전달 의료기기, 안구...",2026-03-31,12월,"김건호, 최의경",http://recensmedical.co.kr/,경기도


In [287]:
columns = code_all[0].columns

In [288]:
code_name_stock_df = code_all[0][[columns[0], columns[2]]]

In [289]:
code_name_stock_df.columns = ['names', 'code']

In [290]:
code_name_stock_df['change_price'] = np.nan

/tmp/ipykernel_5301/1402249009.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  code_name_stock_df['change_price'] = np.nan


In [291]:
code_name_stock_df

,names,code,change_price
0,보원케미칼,0010F0,NaN
1,교보20호스팩,0132G0,NaN
2,인벤테라,0007J0,NaN
3,신한제17호스팩,0130D0,NaN
4,리센스메디컬,394420,NaN
...,...,...,...
2764,유한양행,000100,NaN
2765,CJ대한통운,000120,NaN
2766,경방,000050,NaN
2767,유수홀딩스,000700,NaN


In [292]:
import pandas_datareader.data as web

target_code 는 code_name_stock_df 에서 넣어주게하고

strart ,end 설정해서

주가 가져오기

In [293]:
code_name_stock_df.code[100]

'125020'

In [294]:
stock = web.DataReader(code_name_stock_df.code[100], 'naver', '2026-01-10', '2026-04-12')

In [295]:
stock

,Open,High,Low,Close,Volume
Date,,,,,
2026-01-12,4000,4150,3995,4125,755705
2026-01-13,4125,4160,4000,4005,543496
2026-01-14,4005,4110,3945,4030,462865
2026-01-15,4000,4010,3895,3965,510312
2026-01-16,3970,4050,3925,4000,443590
...,...,...,...,...,...
2026-04-06,4670,4790,4565,4605,426779
2026-04-07,4655,4930,4605,4615,877666
2026-04-08,4865,5030,4805,5000,813670


In [296]:
# 강사님 버전
from datetime import datetime, date, timedelta
import requests
import calendar

end = datetime.now() - timedelta(days=1) # 주가 불러올 때 마지막 시간
start = datetime.now() - timedelta(days=100)
print('start :', start)
print('end :', end)
target_code = '000100'
print('target_code :', target_code)

stock = web.DataReader(target_code, 'naver', start, end)
stock

start : 2026-01-03 06:49:45.618696
end : 2026-04-12 06:49:45.618562
target_code : 000100


,Open,High,Low,Close,Volume
Date,,,,,
2026-01-05,113400,115500,112100,114500,362540
2026-01-06,114700,115900,113400,114500,330145
2026-01-07,113600,114300,112200,113200,342445
2026-01-08,112900,115200,112500,113300,574070
2026-01-09,113700,113800,112200,113300,270683
...,...,...,...,...,...
2026-04-06,92800,93500,91700,91800,181773
2026-04-07,92000,93500,90900,91400,168418
2026-04-08,94400,95400,93500,94200,205545


In [297]:
change_price = int(stock.iloc[-1, 3]) - int(stock.iloc[0, 3])

In [298]:
change_price

-21600

In [299]:
def start_end():
    end = datetime.now() - timedelta(days=1)
    start = datetime.now() - timedelta(days=100)
    return start, end

In [300]:
def get_change_price(df):
    start, end = start_end()
    for i, j in enumerate(df.code):
        target_code = j
        stock = web.DataReader(target_code, 'naver', start, end)
        change_price = int(stock.iloc[-1, 3]) - int(stock.iloc[0, 3])
        df.iloc[i, 2] = change_price
    return df

In [301]:
df = get_change_price(code_name_stock_df) # 이거 40분 걸림 일단 주석처리

open 시가

High 고가

Low 저가

Close 종가

Volume 거래량

In [302]:
df.head()

,names,code,change_price
0,보원케미칼,0010F0,-610.0
1,교보20호스팩,0132G0,-10.0
2,인벤테라,0007J0,-10000.0
3,신한제17호스팩,0130D0,-25.0
4,리센스메디컬,394420,3650.0


In [303]:
df.to_csv('stock_name_code_ChangePrice')

In [304]:
df_10 = df.sort_values(by='change_price', ascending=False)[:10]

In [305]:
# SK하이닉스 기업실적분석 테이블 가지고 오기
SK_hy = pd.read_html('https://finance.naver.com/item/main.naver?code=000660', header=0, encoding='euc-kr')

In [306]:
type(SK_hy)

list

In [307]:
len(SK_hy)

14

In [308]:
SK_hy[4]

,주요재무정보,최근 연간 실적,최근 연간 실적.1,최근 연간 실적.2,최근 연간 실적.3,최근 분기 실적,최근 분기 실적.1,최근 분기 실적.2,최근 분기 실적.3,최근 분기 실적.4,최근 분기 실적.5
0,주요재무정보,2023.12,2024.12,2025.12,2026.12(E),2024.12,2025.03,2025.06,2025.09,2025.12,2026.03(E)
1,주요재무정보,IFRS연결,IFRS연결,IFRS연결,IFRS연결,IFRS연결,IFRS연결,IFRS연결,IFRS연결,IFRS연결,IFRS연결
2,매출액,327657,661930,971467,2674578,197670,176391,222320,244489,328267,496756
3,영업이익,-77303,234673,472063,1944330,80828,74405,92129,113834,191696,345381
4,당기순이익,-91375,197969,429479,1555570,80065,81082,69962,125975,152460,254540
5,영업이익률,-23.59,35.45,48.59,72.70,40.89,42.18,41.44,46.56,58.40,69.53
6,순이익률,-27.89,29.91,44.21,58.16,40.50,45.97,31.47,51.53,46.44,51.24
7,ROE(지배주주),-15.61,31.06,44.15,80.40,31.06,37.94,39.27,43.20,44.15,NaN
8,부채비율,87.52,62.15,45.95,NaN,62.15,52.24,48.13,48.43,45.95,NaN
9,당좌비율,75.97,113.24,132.97,NaN,113.24,104.14,117.87,134.90,132.97,NaN


In [309]:
sk = SK_hy[4]

In [310]:
sk

,주요재무정보,최근 연간 실적,최근 연간 실적.1,최근 연간 실적.2,최근 연간 실적.3,최근 분기 실적,최근 분기 실적.1,최근 분기 실적.2,최근 분기 실적.3,최근 분기 실적.4,최근 분기 실적.5
0,주요재무정보,2023.12,2024.12,2025.12,2026.12(E),2024.12,2025.03,2025.06,2025.09,2025.12,2026.03(E)
1,주요재무정보,IFRS연결,IFRS연결,IFRS연결,IFRS연결,IFRS연결,IFRS연결,IFRS연결,IFRS연결,IFRS연결,IFRS연결
2,매출액,327657,661930,971467,2674578,197670,176391,222320,244489,328267,496756
3,영업이익,-77303,234673,472063,1944330,80828,74405,92129,113834,191696,345381
4,당기순이익,-91375,197969,429479,1555570,80065,81082,69962,125975,152460,254540
5,영업이익률,-23.59,35.45,48.59,72.70,40.89,42.18,41.44,46.56,58.40,69.53
6,순이익률,-27.89,29.91,44.21,58.16,40.50,45.97,31.47,51.53,46.44,51.24
7,ROE(지배주주),-15.61,31.06,44.15,80.40,31.06,37.94,39.27,43.20,44.15,NaN
8,부채비율,87.52,62.15,45.95,NaN,62.15,52.24,48.13,48.43,45.95,NaN
9,당좌비율,75.97,113.24,132.97,NaN,113.24,104.14,117.87,134.90,132.97,NaN


In [311]:
sk_eps = sk.iloc[11,0:5]

In [312]:
sk_eps

,11
주요재무정보,EPS(원)
최근 연간 실적,-12517
최근 연간 실적.1,27182
최근 연간 실적.2,58955
최근 연간 실적.3,220159


In [313]:
code = '000660'

In [314]:
url = f'https://finance.naver.com/item/main.naver?code={code}'

In [315]:
from io import StringIO # 위 방식대로 해도 되지만 이 방식은 에러가 덜 나옴
financials = pd.read_html(StringIO(requests.get(url).text))

In [316]:
financials

[                          0  \
 0  전일  1,027,000  1,027,000   
 1        시가  999,000999,000   
 
                                                    1  \
 0  고가  1,043,0001,043,000  (상한가  1,335,0001,335,0...   
 1               저가  996,000996,000  (하한가  719,000  )   
 
                                 2  
 0       거래량  2,769,188  2,769,188  
 1  거래대금  2,846,084  2,846,084  백만  ,
                           0  \
 0  전일  1,027,000  1,027,000   
 1    시가  1,005,0001,005,000   
 
                                                    1  \
 0  고가  1,043,0001,043,000  (상한가  1,335,0001,335,0...   
 1               저가  984,000984,000  (하한가  719,000  )   
 
                                 2  
 0       거래량  2,150,412  2,150,412  
 1  거래대금  2,179,687  2,179,687  백만  ,
      매도상위       거래량     매수상위       거개량
 0     NaN       NaN      NaN       NaN
 1    삼성증권  315606.0    골드만삭스  385942.0
 2   골드만삭스  286742.0     메릴린치  231902.0
 3    키움증권  275384.0     KB증권  210742.0
 4    KB증권  232897.0   신한투자증권  198

In [317]:
financials[4]

주요재무정보   최근 연간 실적                                     최근 분기 실적  \
       주요재무정보    2023.12    2024.12    2025.12  2026.12(E)    2024.12   
       주요재무정보     IFRS연결     IFRS연결     IFRS연결      IFRS연결     IFRS연결   
0         매출액  327657.00  661930.00  971467.00  2674578.00  197670.00   
1        영업이익  -77303.00  234673.00  472063.00  1944330.00   80828.00   
2       당기순이익  -91375.00  197969.00  429479.00  1555570.00   80065.00   
3       영업이익률     -23.59      35.45      48.59       72.70      40.89   
4        순이익률     -27.89      29.91      44.21       58.16      40.50   
5   ROE(지배주주)     -15.61      31.06      44.15       80.40      31.06   
6        부채비율      87.52      62.15      45.95         NaN      62.15   
7        당좌비율      75.97     113.24     132.97         NaN     113.24   
8         유보율    1397.12    1911.20    3158.59         NaN    1911.20   
9      EPS(원)  -12517.00   27182.00   58955.00   220159.00   10990.00   
10     PER(배)     -11.30       6.40      11.04        4.66       6.40   
11     BPS(원)   77752.00  107256.00  174538.00   391803.00  107256.00   
12     PBR(배)       1.82       1.62       3.73        2.62       1.62   
13   주당배당금(원)    1200.00    2204.00    3000.00     4055.00    1304.00   
14   시가배당률(%)       0.85       1.27       0.46         NaN       0.75   
15    배당성향(%)      -9.06       7.68       4.88         NaN      11.25   

                                                           
      2025.03    2025.06    2025.09    2025.12 2026.03(E)  
       IFRS연결     IFRS연결     IFRS연결     IFRS연결     IFRS연결  
0   176391.00  222320.00  244489.00  328267.00  496756.00  
1    74405.00   92129.00  113834.00  191696.00  345381.00  
2    81082.00   69962.00  125975.00  152460.00  254540.00  
3       42.18      41.44      46.56      58.40      69.53  
4       45.97      31.47      51.53      46.44      51.24  
5       37.94      39.27      43.20      44.15        NaN  
6       52.24      48.13      48.43      45.95        NaN  
7      104.14     117.87     134.90     132.97        NaN  
8     2114.39    2298.84    2636.87    3158.59        NaN  
9    11136.00    9612.00   17301.00   20906.00   40361.00  
10       5.34       7.37       7.09      11.04      25.45  
11  117947.00  126196.00  144811.00  174538.00        NaN  
12       1.62       2.31       2.40       3.73        NaN  
13     375.00     375.00     375.00    1875.00        NaN  
14       0.20       0.13       0.11       0.29        NaN  
15       3.19       3.70       2.06       8.66        NaN

In [318]:
financials[4].iloc[9, :]

주요재무정보    주요재무정보      주요재무정보      EPS(원)
최근 연간 실적  2023.12     IFRS연결    -12517.0
          2024.12     IFRS연결     27182.0
          2025.12     IFRS연결     58955.0
          2026.12(E)  IFRS연결    220159.0
최근 분기 실적  2024.12     IFRS연결     10990.0
          2025.03     IFRS연결     11136.0
          2025.06     IFRS연결      9612.0
          2025.09     IFRS연결     17301.0
          2025.12     IFRS연결     20906.0
          2026.03(E)  IFRS연결     40361.0
Name: 9, dtype: object

In [345]:
df_dict = {}
for i, j in df_10.iterrows():
    url = f'https://finance.naver.com/item/main.naver?code={j.code}'
    fis = pd.read_html(StringIO(requests.get(url).text))
    df_dict[j.names] = fis[4].iloc[9, :]

    df = fis[4].iloc[9, :]

    if len(df) < 4:
        continue

    df.name = j.names
    df_dict[j.code] = df
    print(df_dict, df_10)

{'효성중공업': 주요재무정보    주요재무정보      주요재무정보     EPS(원)
최근 연간 실적  2023.12     IFRS연결    12438.0
          2024.12     IFRS연결    23876.0
          2025.12     IFRS연결    55755.0
          2026.12(E)  IFRS연결    84943.0
최근 분기 실적  2024.12     IFRS연결     9766.0
          2025.03     IFRS연결      10956
          2025.06     IFRS연결       9922
          2025.09     IFRS연결      16109
          2025.12     IFRS연결    18769.0
          2026.03(E)  IFRS연결    14108.0
Name: 9, dtype: object, '298040': 주요재무정보    주요재무정보      주요재무정보     EPS(원)
최근 연간 실적  2023.12     IFRS연결    12438.0
          2024.12     IFRS연결    23876.0
          2025.12     IFRS연결    55755.0
          2026.12(E)  IFRS연결    84943.0
최근 분기 실적  2024.12     IFRS연결     9766.0
          2025.03     IFRS연결      10956
          2025.06     IFRS연결       9922
          2025.09     IFRS연결      16109
          2025.12     IFRS연결    18769.0
          2026.03(E)  IFRS연결    14108.0
Name: 효성중공업, dtype: object}           names    code  change_price
888       

In [334]:
df_dict

{'효성중공업': 주요재무정보    주요재무정보      주요재무정보     EPS(원)
 최근 연간 실적  2023.12     IFRS연결    12438.0
           2024.12     IFRS연결    23876.0
           2025.12     IFRS연결    55755.0
           2026.12(E)  IFRS연결    84943.0
 최근 분기 실적  2024.12     IFRS연결     9766.0
           2025.03     IFRS연결      10956
           2025.06     IFRS연결       9922
           2025.09     IFRS연결      16109
           2025.12     IFRS연결    18769.0
           2026.03(E)  IFRS연결    14108.0
 Name: 9, dtype: object,
 '298040': 주요재무정보    주요재무정보      주요재무정보     EPS(원)
 최근 연간 실적  2023.12     IFRS연결    12438.0
           2024.12     IFRS연결    23876.0
           2025.12     IFRS연결    55755.0
           2026.12(E)  IFRS연결    84943.0
 최근 분기 실적  2024.12     IFRS연결     9766.0
           2025.03     IFRS연결      10956
           2025.06     IFRS연결       9922
           2025.09     IFRS연결      16109
           2025.12     IFRS연결    18769.0
           2026.03(E)  IFRS연결    14108.0
 Name: 효성중공업, dtype: object,
 '본시스템즈': 매도잔량         Na

In [385]:
df = df_dict['298040'][2]

In [386]:
df

,level_1,효성중공업
5,2024-12-31,9766.0
6,2025-03-31,10956
7,2025-06-30,9922
8,2025-09-30,16109
9,2025-12-31,18769.0
10,2026-03-31,14108.0


In [387]:
dfs = df_dict.copy()

In [388]:
for i in df.index:
    str_date = str(df.loc[i].level_1)
    year = int(str_date[:4])
    month = int(str_date[5:7])
    fd, last_day = calendar.monthrange(year, month)
    # print(year, month, last_day, fd)
    dates = date(year, month, last_day)
    print(dates)
    df.loc[i, 'level_1'] = dates
print(df)

2024-12-31
2025-03-31
2025-06-30
2025-09-30
2025-12-31
2026-03-31
       level_1    효성중공업
5   2024-12-31   9766.0
6   2025-03-31    10956
7   2025-06-30     9922
8   2025-09-30    16109
9   2025-12-31  18769.0
10  2026-03-31  14108.0


In [378]:
def select_figure(df):
    name = df.columns[-1]
    df_value = pd.to_numeric(df[name])
    if len(df_value) == len(df_value[df_value>0]):
        return True
    return False

In [379]:
def date_transform(df):
    for i in df.index:
        str_date = df.loc[i].level_1
        year = int(str_date[:4])
        month = int(str_date[5:7])
        fd, last_day = calendar.monthrange(year, month)
        dates = date(year, month, last_day)
        df.loc[i,'level_1']=dates
    return df

In [390]:
for i in dfs:
    df = dfs[i]
    df = df.dropna()
    df = df.reset_index()
    name = df.columns[3]
    df = df[df.level_0=='최근 분기 실적'][['level_1', name]]
    df = date_transform(df)
    start = date(df.iloc[0, 0].year, df.iloc[0, 0].month, 1)
    end = df.iloc[-1, 0]
    df_dict[i] = [start, end, df]

AttributeError: 'DataFrame' object has no attribute 'level_0'

In [381]:
df.iloc[0, 0].year, df.iloc[0, 0].month

AttributeError: 'list' object has no attribute 'iloc'